# Fine-tune Llama 3.2 3B with QLoRA — Contract Clause Classifier

Realistic on a single Colab A100 (40GB). This fine-tunes a small set of
LoRA adapter weights on top of a frozen, 4-bit quantized base model —
not full-parameter training, which would not fit in memory.

**Before running for real:** replace the sample data in the dataset
cell below with hundreds of real labeled examples from your actual
ContractorCloud documents. 10 examples (as shipped here) will run
end-to-end and prove the pipeline works, but won't produce a model
good enough to actually use.

## 1. Runtime check — confirm you actually have a GPU

In [1]:
!nvidia-smi

# If this errors or shows no GPU: Runtime -> Change runtime type -> GPU (A100 if available on your plan)

Fri Jul  3 03:44:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Install dependencies

In [2]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.1 MB/s eta 0:00:00


## 3. Authenticate with Hugging Face (Llama models are gated)

In [ ]:
from huggingface_hub import login
import getpass

# Get a token at huggingface.co/settings/tokens, and request access to
# meta-llama/Llama-3.2-3B-Instruct at huggingface.co/meta-llama (approval
# is usually fast, sometimes instant)
login(getpass.getpass("Hugging Face token: "))

## 4. Load your real training data

This loads `training_data.jsonl`, built by running
`02_build_training_set.py` on your real labeled clauses
(from `label_tool.py`). Upload that file to this Colab session
first (the folder icon on the left sidebar -> upload), or mount
Google Drive if your data lives there.

In [ ]:
import json

DATA_PATH = "training_data.jsonl"  # upload this file to the Colab session first

formatted_data = []
with open(DATA_PATH) as f:
    for line in f:
        formatted_data.append(json.loads(line))

print(f"Loaded {len(formatted_data)} real training examples")
print()
print("--- Sample ---")
print(formatted_data[0]["text"])

if len(formatted_data) < 200:
    print()
    print(f"WARNING: only {len(formatted_data)} examples. This will train and ")
    print("run end-to-end, but is too small to produce a reliable model. ")
    print("Run 02_build_training_set.py's coverage report and label more ")
    print("before trusting this model's output.")

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
print(dataset)

In [ ]:
# Label list — must match taxonomy.py CATEGORY_IDS exactly.
# Upload taxonomy.py to this Colab session alongside training_data.jsonl,
# then this import works. Or keep the hardcoded list below as a fallback.
try:
    from taxonomy import CATEGORY_IDS
    LABELS = CATEGORY_IDS
except ImportError:
    LABELS = [
        "insurance_general_liability",
        "insurance_additional_insured",
        "insurance_workers_comp",
        "insurance_auto_liability",
        "insurance_certificate_timing",
        "payment_terms_timing",
        "payment_retainage",
        "payment_pay_if_paid",
        "indemnification_scope",
        "limitation_of_liability",
        "scope_of_work",
        "schedule_milestones",
        "change_orders",
        "termination_for_cause",
        "termination_notice_period",
        "dispute_resolution",
        "lien_waivers",
    ]
print(f"LABELS defined: {len(LABELS)} categories")


## 5. Load the base model in 4-bit (this is what makes a 3B model fit comfortably, and is the same technique that lets you go up to 7B-13B on a single A100)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Loaded {MODEL_NAME} in 4-bit. GPU memory used:")
print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6. Attach LoRA adapters — this is the part that actually trains

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect roughly: trainable params ~11M out of ~3.2B total, around 0.3-0.4%
# This is the entire point of LoRA -- you're training under half a percent
# of the parameters, which is why this fits and runs fast even on one GPU.

## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./clause-classifier-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

## 8. Test the fine-tuned model on a new example

In [ ]:
test_clause = "Subcontractor shall provide a certificate naming the Owner and Contractor as additional insureds within 10 days of contract execution."

prompt = f"Classify the following contract clause into exactly one category: {', '.join(LABELS)}.\n\nClause: \"{test_clause}\"\n\nCategory:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Predicted category:{result}")
print(f"(Expected: insurance_requirement)")

## 9. Save the model properly

"Properly" means three things, not just calling `save_pretrained`:

1. **Version it** -- a timestamp or version tag, so you can tell
   which model is which once you've trained several iterations
2. **Document it** -- a model card recording what data trained it,
   how many examples, which categories, and the eval results --
   without this, a model from two months from now is just a
   mystery folder of numbers
3. **Save the adapter, not the full model** -- LoRA adapters are a
   few MB; re-merging them onto the base model happens at load time,
   so you never need to save/download a multi-GB file

In [ ]:
import datetime
import json
import subprocess

VERSION = datetime.datetime.now().strftime("%Y%m%d-%H%M")
MODEL_DIR = f"clause-classifier-lora-v{VERSION}"

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Model card -- the actual documentation that makes this model usable
# by you (or anyone else) months from now without having to remember
# everything about this training run.
model_card = {
    "version": VERSION,
    "base_model": MODEL_NAME,
    "task": "Contract clause classification + compliance check",
    "num_training_examples": len(formatted_data),
    "categories": LABELS if 'LABELS' in dir() else "see taxonomy.py",
    "training_args": {
        "epochs": training_args.num_train_epochs,
        "learning_rate": training_args.learning_rate,
        "lora_r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
    },
    "trained_on": datetime.datetime.now().isoformat(),
}

with open(f"{MODEL_DIR}/model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)

print(f"Saved to {MODEL_DIR}/")
print(json.dumps(model_card, indent=2))

!zip -rq {MODEL_DIR}.zip {MODEL_DIR}
print(f"\nZipped: {MODEL_DIR}.zip")

from google.colab import files
files.download(f"{MODEL_DIR}.zip")

### Also save to Google Drive (recommended, more durable than relying on the download)

Colab's local disk is wiped when the session ends or times out. If you
navigate away before the download finishes, you lose the model. Saving
a copy to Drive first removes that risk entirely.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DEST = f"/content/drive/MyDrive/contractorcloud-models/{MODEL_DIR}"
shutil.copytree(MODEL_DIR, DRIVE_DEST)
print(f"Also saved to Google Drive at: {DRIVE_DEST}")

## 10. Using this adapter later (e.g. inside an MCP tool)

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", load_in_4bit=True, device_map="auto")
model = PeftModel.from_pretrained(base, "./clause-classifier-lora-final")
tokenizer = AutoTokenizer.from_pretrained("./clause-classifier-lora-final")
```

This is exactly the kind of model `model_router_mcp`'s `call_model` function
could route to as a third, even-cheaper tier — below Nemotron, for a task
this narrow and specific.